# Reproduce the study: compartment-specific transcriptomic clocks in progestin-exposed endometrium

Run all cells from top to bottom (**Kernel > Restart & Run All**). The notebook

1. downloads every raw input (about 28 GB) and checks it against `inputs/data_manifest.json`,
2. runs the analysis scripts S1 to S13 in order, each logged to `$WORK/logs/`,
3. rebuilds the figures,
4. runs the unit tests,
5. compares every regenerated number with the version committed in git.

Requirements: Python packages in `requirements.txt` and `curl`.
Compute: CPU only, at most 16 cores, about 30 GB of RAM at the peak, about 1 hour after the download.

## 0. Settings

In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Where derived files and small external downloads go, and where the two HECA h5ad files live (16 GB and 9 GB).
WORK = Path(os.environ.get("WORK", ROOT / "work"))
HECA_DIR = Path(os.environ.get("HECA_DIR", ROOT / "data" / "heca"))
WORKERS = 8          # process pool size for the h5ad passes
THREADS = 16         # BLAS threads for the model fits
CHECKSUM = True      # SHA-256 of every input (about 2 minutes for the large files)

os.environ.update(WORK=str(WORK), HECA_DIR=str(HECA_DIR))
(WORK / "logs").mkdir(parents=True, exist_ok=True)
PY = sys.executable
print("repository:", ROOT); print("WORK:", WORK); print("HECA_DIR:", HECA_DIR); print("python:", PY)

In [ ]:
def run(name, args, threads=THREADS):
    """Run one pipeline step from scripts/, log to $WORK/logs/<name>.log, stop the notebook on failure."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), OPENBLAS_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    log = WORK / "logs" / f"{name}.log"; t = time.time()
    with open(log, "w") as fh:
        r = subprocess.run([PY] + args, cwd=ROOT / "scripts", env=env, stdout=fh, stderr=subprocess.STDOUT)
    print(f"{name:<28} {'ok' if r.returncode == 0 else 'FAILED'}  {time.time() - t:7.1f} s   log: {log}")
    if r.returncode:
        print(open(log).read()[-3000:]); raise RuntimeError(f"step {name} failed")

def show_json(path, keys=None):
    d = json.load(open(ROOT / "outputs" / path))
    print(json.dumps({k: d[k] for k in keys} if keys else d, indent=1)[:3000])

## 1. Raw data

HECA (Reproductive Cell Atlas), Huang 2026 (CELLxGENE), GEO series matrices with platform annotations, and PGR ChIP-seq bigWigs (GSE132712) with hg19 RefSeq TSSs.

In [ ]:
run("fetch_data", ["fetch_data.py"] + ([] if CHECKSUM else ["--no-checksum"]), threads=1)
print(open(WORK / "logs" / "fetch_data.log").read())

## 2. Donor x compartment profiles (S1)

In [ ]:
run("s1_profiles_nuclei", ["s1_profiles.py", "nuclei", "--workers", str(WORKERS)], threads=1)
run("s1_profiles_cells", ["s1_profiles.py", "cells", "--workers", str(WORKERS)], threads=1)

## 3. Cycle clocks, positive controls, H1 to H3 (S2) and endometriosis and age (S3)

In [ ]:
run("s2_nuclei", ["s2_clock.py", "nuclei"])
run("s2_cells", ["s2_clock.py", "cells"])
run("s2_cells_combined_pill", ["s2_clock.py", "cells", "combined_pill"])
run("s3_endometriosis_age", ["s3_endometriosis_age.py"])
show_json("s2_nuclei_summary.json", ["pc1", "pc2", "h1"])

## 4. Sensitivity (S4), mechanism (S5)

In [ ]:
for s, g in (("nuclei", []), ("cells", ["combined_pill"])):
    run(f"s4_{s}", ["s4_sensitivity.py", s] + g)
    run(f"s5_{s}", ["s5_mechanism.py", s] + g, threads=4)
show_json("s4_nuclei_summary.json", ["main"])

## 5. External validation: programmed cycles (S6), bulk cohorts (S7), effect sizes (S8)

In [ ]:
run("s6_external_huang", ["s6_external_huang.py"])
run("s7_external_bulk", ["s7_external_bulk.py"])
run("s8_effects", ["s8_effects.py"])
show_json("s6_huang_summary.json", ["unscaled", "primary"])

## 6. Robustness analyses R1 to R18 (S9 to S15, post hoc)

In [ ]:
run("s9_review", ["s9_review.py"])
run("s10_rank_bulk", ["s10_rank_bulk.py"])
run("s11_cell_level", ["s11_cell_level.py", "--workers", str(WORKERS), "--draws", "20"], threads=2)
run("s12_sublineage", ["s12_sublineage.py", "--workers", str(WORKERS)], threads=2)
run("s13_rank_huang_deconv", ["s13_rank_huang_deconv.py"])
run("s14_pgr_cistrome", ["s14_pgr_cistrome.py"], threads=1)
run("s15_clock_rigor", ["s15_clock_rigor.py"])
show_json("s12_sublineage.json")

## 7. Figures

In [ ]:
run("figures", ["figures.py"])
from IPython.display import Image, display
for k in ("1", "2", "3", "4"):
    display(Image(filename=str(ROOT / "figures" / f"figure{k}.png"), width=900))

## 8. Key results: effect sizes by cohort

In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / "outputs" / "s8_effects.csv"))

## 9. Unit tests

In [ ]:
r = subprocess.run([PY, "-m", "pytest", "-q", "tests"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-1500:]); assert r.returncode == 0

## 10. Reproducibility check

Every number in `outputs/` is compared with the committed version (`git show HEAD:outputs/...`). JSON and CSV values
must agree to 6 significant digits. Any difference is listed.

In [ ]:
import io, math
import pandas as pd

def committed(rel):
    r = subprocess.run(["git", "show", f"HEAD:{rel}"], cwd=ROOT, capture_output=True, text=True)
    return r.stdout if r.returncode == 0 else None

def flat(x, p=""):
    if isinstance(x, dict):
        for k, v in x.items(): yield from flat(v, f"{p}.{k}")
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from flat(v, f"{p}[{i}]")
    else:
        yield p, x

def same(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
        return (math.isnan(a) and math.isnan(b)) or math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    return a == b

diffs, checked = [], 0
for f in sorted((ROOT / "outputs").iterdir()):
    old = committed(f"outputs/{f.name}")
    if old is None:
        diffs.append((f.name, "not in git")); continue
    if f.suffix == ".json":
        a, b = dict(flat(json.loads(f.read_text()))), dict(flat(json.loads(old)))
        for k in set(a) | set(b):
            checked += 1
            if not same(a.get(k), b.get(k)): diffs.append((f.name, k, a.get(k), b.get(k)))
    else:
        a, b = pd.read_csv(f), pd.read_csv(io.StringIO(old))
        checked += a.size
        if a.shape != b.shape or not all(a.columns == b.columns):
            diffs.append((f.name, "shape or columns differ")); continue
        num = a.select_dtypes("number").columns
        bad = ~((a[num] - b[num]).abs() <= 1e-6 * b[num].abs().clip(lower=1e-3)) & ~(a[num].isna() & b[num].isna())
        if bad.values.any() or not a.drop(columns=num).fillna("").equals(b.drop(columns=num).fillna("")):
            diffs.append((f.name, f"{int(bad.values.sum())} numeric cells differ"))
print(f"values checked: {checked:,}")
print("REPRODUCED: all outputs match the committed results" if not diffs else f"{len(diffs)} differences:")
for d in diffs[:50]: print("  ", d)